# PARSER-004 - Dataset dan model kategori TF-IDF Logistic Regression

Notebook ini membantu memahami dataset kategori awal, melatih model TF-IDF + Logistic Regression, menyimpan model dengan joblib, dan mencoba prediksi kategori utama. Training production juga memakai normalisasi keyword dan augmentasi frasa agar model lebih tahan terhadap variasi input chat.

Notebook memakai file production:
- `backend/app/modules/parser/data/category_dataset.csv`
- `backend/app/modules/parser/train_model.py`
- `backend/app/modules/parser/category_classifier.py`

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BACKEND = ROOT / "backend"
sys.path.insert(0, str(BACKEND))

DATASET_PATH = BACKEND / "app" / "modules" / "parser" / "data" / "category_dataset.csv"
MODEL_PATH = BACKEND / "app" / "modules" / "parser" / "models" / "category_classifier.joblib"
METRICS_PATH = BACKEND / "app" / "modules" / "parser" / "models" / "category_metrics.json"

DATASET_PATH, MODEL_PATH, METRICS_PATH


## Load dan inspeksi dataset

Dataset MVP dibuat kecil tetapi seimbang untuk kategori default: Makanan, Transportasi, Tagihan, Belanja, Hiburan, Kesehatan, Pendidikan, Gaji, Tabungan, dan Lainnya.

In [ ]:
with DATASET_PATH.open("r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print("rows:", len(rows))
print("category distribution:")
print(json.dumps(dict(sorted(Counter(row["category"] for row in rows).items())), indent=2))
rows[:5]


## Train model

Cell ini menjalankan fungsi training production. Model final di-fit ulang ke seluruh dataset setelah evaluasi holdout, lalu disimpan sebagai `category_classifier.joblib`.

In [ ]:
from app.modules.parser.train_model import train_and_save

metrics = train_and_save(
    dataset_path=DATASET_PATH,
    model_path=MODEL_PATH,
    metrics_path=METRICS_PATH,
)
print("raw_dataset_rows:", metrics["dataset_rows"])
print("augmented_training_rows:", metrics["training_rows"])
print("baseline_accuracy:", metrics["baseline_accuracy"])
print("keyword_smoke_accuracy:", metrics["keyword_smoke_accuracy"])
print("model_path:", metrics["model_path"])


## Coba prediksi

Confidence model adalah probabilitas multiclass dari Logistic Regression. Normalizer mengubah keyword umum seperti `ojol`, `pln`, `indihome`, `spp`, `salary`, dan `dividen` menjadi sinyal kategori yang lebih jelas sebelum TF-IDF.

In [ ]:
from app.modules.parser.category_classifier import load_category_model, predict_category

load_category_model.cache_clear()
examples = [
    "beli nasi padang",
    "bayar bensin motor",
    "tagihan internet bulanan",
    "gaji masuk bulan ini",
    "nabung bulanan",
    "beli obat batuk",
]

for text in examples:
    prediction = predict_category(text)
    print(text, "=>", prediction.to_dict() if prediction else None)


## Catatan pengembangan

- Tambahkan data nyata dari log transaksi yang sudah dikoreksi user untuk menaikkan akurasi.
- Pastikan setiap kategori tetap punya jumlah contoh yang seimbang.
- Re-run `python backend/app/modules/parser/train_model.py` setelah dataset berubah.